# Data control panel

Local control panel for checking data freshness, updating the tournament master file, and running the project smoke check.

In [ ]:
from __future__ import annotations

import html as html_module
import os
import subprocess
import sys
import threading
from pathlib import Path

from IPython.display import Markdown, display
import ipywidgets as widgets


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not find project root.')


def resolve_project_python(project_root: Path) -> str:
    candidates = [
        project_root / '.venv' / 'Scripts' / 'python.exe',
        project_root / 'venv' / 'Scripts' / 'python.exe',
        project_root / '.venv' / 'bin' / 'python',
        project_root / 'venv' / 'bin' / 'python',
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return sys.executable


PROJECT_ROOT = find_project_root()
PYTHON = resolve_project_python(PROJECT_ROOT)
BUTTONS: list[widgets.Button] = []
LOG_LINES: list[str] = []
MAX_LOG_LINES = 1000


def set_buttons_disabled(disabled: bool) -> None:
    for button in BUTTONS:
        button.disabled = disabled


def set_log(text: str) -> None:
    log_area.value = text


def append_log(text: str) -> None:
    for line in text.splitlines() or ['']:
        LOG_LINES.append(line)
    del LOG_LINES[:-MAX_LOG_LINES]
    log_area.value = '\n'.join(LOG_LINES)


def describe_output_line(line: str, title: str) -> str | None:
    text = line.strip()
    if not text:
        return None
    if text.startswith('$ '):
        return 'Launching command'
    if text.startswith('===') and text.endswith('==='):
        return text.strip('= ').strip()
    if text.startswith('Full RTT predictor pipeline'):
        return 'Full pipeline started'
    if text.startswith('Preflight dependency check'):
        return 'Checking Python dependencies'
    if text.startswith('Missing Python packages:'):
        return text
    if text.startswith('Python executable:'):
        return text
    if text.startswith('Project root:'):
        return 'Checking project location'
    if text.startswith('Started at:'):
        return 'Preparing pipeline steps'
    if text.startswith('Opening RTT calendar'):
        return text
    if text.startswith('Warning: RTT calendar update failed'):
        return text
    if text.startswith('Parsing calendar age group:'):
        return 'Calendar filter: ' + text.split(':', 1)[1].strip()
    if text.startswith('Fetching tournament details'):
        return text
    if text.startswith('Tournament download plan:'):
        return text
    if text.startswith('future_start_date') or text.startswith('cached_html') or text.startswith('completed_cached') or text.startswith('cancelled'):
        return 'Skipping/caching: ' + text
    if text.startswith('Updated matches_page_saved'):
        return text
    if text.startswith('Parsed tournaments:'):
        return text
    if text.startswith('Master before:') or text.startswith('Master after:'):
        return text
    if text.startswith('New tournaments:'):
        return text
    if text.startswith('Output:'):
        return 'Writing output file: ' + text.split(':', 1)[1].strip()
    if text.startswith('Data status'):
        return 'Reading local data status'
    if text.startswith('Rankings rows:'):
        return 'Checking rankings dataset'
    if text.startswith('Matches rows:'):
        return 'Checking parsed matches dataset'
    if text.startswith('Dataset rows:'):
        return 'Checking model dataset'
    if text.startswith('Wrote') and 'data_manifest' in text:
        return 'Updating data manifest'
    if text.startswith('notebook ok:'):
        return 'Verifying notebook: ' + text.split(':', 1)[1].strip()
    if text.startswith('dataset ok'):
        return 'Verifying final dataset'
    if text.startswith('tournaments ok'):
        return 'Verifying tournament master'
    if text.startswith('project verification passed'):
        return 'Project verification passed'
    if text.startswith('Step failed:'):
        return text
    if 'ModuleNotFoundError' in text:
        return text
    return None


def set_current_step(text: str) -> None:
    current_step_html.value = '<b>Now:</b> ' + html_module.escape(text)


def run_project_command_streaming(title: str, args: list[str]) -> None:
    set_buttons_disabled(True)
    progress.value = 0
    progress.bar_style = 'info'
    status_html.value = f'<b>{title}</b>: running...'
    set_current_step('Starting ' + title)

    command = [PYTHON, '-u', *args]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONUTF8'] = '1'

    LOG_LINES.clear()
    set_log('')
    append_log(f'### {title}')
    append_log('$ ' + ' '.join(command))
    append_log('')

    try:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )

        line_count = 0
        assert process.stdout is not None
        for line in process.stdout:
            line_count += 1
            progress.value = min(99, 5 + line_count)
            current_description = describe_output_line(line, title)
            if current_description:
                set_current_step(current_description)
            append_log(line.rstrip('\n'))

        return_code = process.wait()
        if return_code == 0:
            progress.value = 100
            progress.bar_style = 'success'
            status_html.value = f'<b>{title}</b>: done'
            set_current_step('Completed ' + title)
        else:
            progress.value = 100
            progress.bar_style = 'danger'
            status_html.value = f'<b>{title}</b>: failed with code {return_code}'
            set_current_step(f'Stopped after error code {return_code}')
    except Exception as exc:
        progress.value = 100
        progress.bar_style = 'danger'
        status_html.value = f'<b>{title}</b>: failed'
        set_current_step('Stopped after an unexpected error')
        append_log(str(exc))
    finally:
        set_buttons_disabled(False)


def show_command(title: str, args: list[str]) -> None:
    thread = threading.Thread(target=run_project_command_streaming, args=(title, args), daemon=True)
    thread.start()


status_button = widgets.Button(description='Refresh status', button_style='info')
tournaments_button = widgets.Button(description='Update tournaments master', button_style='warning')
calendar_button = widgets.Button(description='Update RTT calendar', button_style='warning')
verify_button = widgets.Button(description='Verify project', button_style='success')
pipeline_button = widgets.Button(description='Run full pipeline', button_style='danger')
BUTTONS.extend([status_button, tournaments_button, calendar_button, pipeline_button, verify_button])

progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='')
status_html = widgets.HTML(value='Ready')
current_step_html = widgets.HTML(value='<b>Now:</b> idle')
log_area = widgets.Textarea(
    value='',
    disabled=True,
    layout=widgets.Layout(width='100%', height='360px'),
)

status_button.on_click(lambda _: show_command('Data status', ['scripts/data_status.py', '--write-manifest']))
tournaments_button.on_click(lambda _: show_command('Tournament master update', ['scripts/bootstrap_tournaments_master.py']))
calendar_button.on_click(lambda _: show_command('RTT calendar update', ['scripts/parse_rtt_calendar.py']))
pipeline_button.on_click(lambda _: show_command('Full update pipeline', ['scripts/run_full_pipeline.py', '--continue-on-calendar-error']))
verify_button.on_click(lambda _: show_command('Project verification', ['scripts/verify_project.py']))

display(Markdown(f'Project root: `{PROJECT_ROOT}`'))
display(Markdown(f'Python: `{PYTHON}`'))
display(widgets.HBox([status_button, tournaments_button, calendar_button, pipeline_button, verify_button]))
display(widgets.VBox([status_html, current_step_html, progress, log_area]))
show_command('Data status', ['scripts/data_status.py', '--write-manifest'])


In [ ]:
# Unified prediction command center
from pathlib import Path
import html
import json
import os
import subprocess

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

prediction_output = widgets.Output()
PREDICTION_CLI = PROJECT_ROOT / 'scripts' / 'prediction_cli.py'
AGE_CONTEXT = {
    'unknown': '__UNKNOWN_AGE__',
    'U15': '\u0434\u043e 15 \u043b\u0435\u0442',
    'U17': '\u0434\u043e 17 \u043b\u0435\u0442',
    'U19': '\u0434\u043e 19 \u043b\u0435\u0442',
    'adult': '\u0432\u0437\u0440\u043e\u0441\u043b\u044b\u0435',
}



def _run_prediction_cli(args):
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    completed = subprocess.run(
        [PYTHON, '-u', str(PREDICTION_CLI), *args],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
        env=env,
    )
    if completed.returncode != 0:
        raise RuntimeError((completed.stdout or '') + (completed.stderr or ''))
    return json.loads(completed.stdout)


try:
    players_payload = _run_prediction_cli(['players'])
    player_names = players_payload.get('players', [])
    max_match_date = pd.Timestamp(players_payload.get('max_match_date')).date()
    bundle_status = f"Loaded via project Python: {Path(PYTHON).name}; model: {players_payload.get('model_name')}; max match date: {max_match_date}"
except Exception as exc:
    players_payload = {'ok': False}
    player_names = []
    max_match_date = pd.Timestamp.today().date()
    bundle_status = f"Prediction bundle is not ready: {exc}"

player1_box = widgets.Combobox(
    placeholder='Start typing player 1 surname',
    options=player_names,
    description='Player 1:',
    ensure_option=False,
    layout=widgets.Layout(width='48%'),
)
player2_box = widgets.Combobox(
    placeholder='Start typing player 2 surname',
    options=player_names,
    description='Player 2:',
    ensure_option=False,
    layout=widgets.Layout(width='48%'),
)
prediction_date_picker = widgets.DatePicker(
    description='Date:',
    value=max_match_date,
    layout=widgets.Layout(width='220px'),
)
age_box = widgets.Dropdown(
    options=['unknown', 'U15', 'U17', 'U19', 'adult'],
    value='unknown',
    description='Age:',
    layout=widgets.Layout(width='180px'),
)
predict_button = widgets.Button(description='Predict match', button_style='primary')
reload_bundle_button = widgets.Button(description='Reload bundle', button_style='info')
prediction_status = widgets.HTML(value=f"<b>Prediction:</b> {bundle_status}")



def _to_frame(records):
    return pd.DataFrame(records or [])



def _fmt_date(value):
    if value in [None, ''] or pd.isna(value):
        return ''
    parsed = pd.to_datetime(value, errors='coerce')
    return '' if pd.isna(parsed) else parsed.strftime('%Y-%m-%d')



def _fmt_num(value, digits=1):
    if value in [None, ''] or pd.isna(value):
        return ''
    return f'{float(value):,.{digits}f}'.replace(',', ' ')



def _fmt_int(value):
    if value in [None, ''] or pd.isna(value):
        return ''
    return f'{int(round(float(value))):,}'.replace(',', ' ')



def _fmt_pct(value, signed=False):
    if value in [None, ''] or pd.isna(value):
        return ''
    sign = '+' if signed else ''
    return f'{float(value):{sign}.1%}'


def _fmt_text(value):
    if value in [None, ''] or pd.isna(value):
        return ''
    return str(value)



def _section(title, subtitle=''):
    subtitle_html = f'<div style="color:#666; margin-top:2px;">{html.escape(subtitle)}</div>' if subtitle else ''
    display(HTML(f'''
    <div style="font-family: system-ui; margin: 22px 0 10px; border-top: 1px solid #e6e6e6; padding-top: 14px;">
      <h3 style="margin: 0;">{html.escape(title)}</h3>
      {subtitle_html}
    </div>
    '''))



def _probability_bar(player1_name, player2_name, p1):
    p2 = 1.0 - p1
    return f'''
    <div style="font-family: system-ui; max-width: 980px;">
      <h3 style="margin-bottom: 6px;">{html.escape(player1_name)} vs {html.escape(player2_name)}</h3>
      <div style="font-size: 18px; margin-bottom: 8px;">
        <b>{html.escape(player1_name)}</b>: {p1:.1%} &nbsp; | &nbsp; <b>{html.escape(player2_name)}</b>: {p2:.1%}
      </div>
      <div style="height: 22px; border: 1px solid #ddd; border-radius: 4px; overflow: hidden; display: flex;">
        <div style="width: {p1*100:.2f}%; background: #4f7cff;"></div>
        <div style="width: {p2*100:.2f}%; background: #ff8a4f;"></div>
      </div>
    </div>
    '''



def _profile_card(row, accent):
    player = html.escape(str(row.get('Player', '')))
    fields = [
        ('Rank', _fmt_int(row.get('Rank'))),
        ('Rating points', _fmt_int(row.get('Points'))),
        ('Rating date', _fmt_date(row.get('Rating date'))),
        ('Rating age', _fmt_text(row.get('Rating age'))),
        ('ELO', _fmt_num(row.get('ELO'), 1)),
        ('Matches', _fmt_int(row.get('Matches'))),
        ('Wins', _fmt_int(row.get('Wins'))),
        ('Win rate', _fmt_pct(row.get('Win rate'))),
        ('Form 5', _fmt_pct(row.get('Form 5'))),
        ('Form 10', _fmt_pct(row.get('Form 10'))),
        ('Last match', _fmt_date(row.get('Last match'))),
        ('Rest days', _fmt_int(row.get('Rest days'))),
        ('Avg opponent ELO last 5', _fmt_num(row.get('Avg opp ELO last 5'), 1)),
        ('Head to head', f"{_fmt_int(row.get('H2H wins'))} / {_fmt_int(row.get('H2H matches'))}"),
    ]
    items = ''.join(
        f'''<div style="display:flex; justify-content:space-between; gap:16px; padding:4px 0; border-bottom:1px solid #f0f0f0;">
              <span style="color:#60636b;">{html.escape(label)}</span><b>{html.escape(str(value))}</b>
            </div>'''
        for label, value in fields
    )
    return f'''
    <div style="flex:1; min-width:340px; border:1px solid #e3e5eb; border-radius:8px; overflow:hidden; background:white;">
      <div style="background:{accent}; color:white; padding:10px 12px; font-weight:700;">{player}</div>
      <div style="padding:10px 12px; font-size:13px;">{items}</div>
    </div>
    '''



def _display_player_cards(payload):
    profiles = _to_frame(payload.get('profiles'))
    if profiles.empty:
        return
    cards = ''.join(_profile_card(row, accent) for (_, row), accent in zip(profiles.iterrows(), ['#4f7cff', '#ff8a4f']))
    display(HTML(f'<div style="display:flex; gap:14px; flex-wrap:wrap; max-width:1100px;">{cards}</div>'))



def _format_date_axis(ax):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    for label in ax.get_xticklabels():
        label.set_rotation(35)
        label.set_ha('right')



def _plot_probability_timeline(payload):
    timeline = _to_frame(payload.get('timeline'))
    if timeline.empty:
        display(HTML('<p>Probability history is not available yet: not enough data for one of the players.</p>'))
        return
    timeline['date'] = pd.to_datetime(timeline['date'], errors='coerce')
    timeline = timeline.dropna(subset=['date']).sort_values('date')
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(timeline['date'], timeline['p_player1_win'], marker='o', linewidth=2, color='#4f7cff')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_title(f"Win probability history: {payload.get('player1_name')}")
    ax.set_ylabel('Probability')
    ax.grid(alpha=0.25)
    _format_date_axis(ax)
    plt.tight_layout()
    display(fig)
    plt.close(fig)



def _plot_player_history(match_records, rating_records, title, prediction_date=None):
    match_history = _to_frame(match_records)
    rating_history = _to_frame(rating_records)
    prediction_date = pd.to_datetime(prediction_date, errors='coerce') if prediction_date else pd.NaT
    fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=False)

    if not rating_history.empty:
        rating_history['classification_date'] = pd.to_datetime(
            rating_history['classification_date'],
            errors='coerce',
        )
        rating_history = rating_history.dropna(subset=['classification_date']).sort_values('classification_date')
        if pd.notna(prediction_date):
            rating_history = rating_history[rating_history['classification_date'] <= prediction_date]

        if 'points' in rating_history.columns:
            points_history = (
                rating_history
                .dropna(subset=['points'])
                .sort_values(['classification_date', 'points'])
                .drop_duplicates(subset=['classification_date'], keep='last')
            )
            axes[0].plot(
                points_history['classification_date'],
                points_history['points'],
                marker='o',
                linewidth=1.7,
                color='#5b63c7',
            )

        if 'rank' in rating_history.columns:
            rank_history = rating_history.dropna(subset=['rank'])
            for age_group, group in rank_history.groupby('age_group', dropna=False):
                axes[1].plot(group['classification_date'], group['rank'], marker='o', linewidth=1.4, label=str(age_group))
            if not rank_history.empty:
                axes[1].legend(loc='best')
                axes[1].invert_yaxis()

    axes[0].set_title(f'{title}: rating points')
    axes[0].grid(alpha=0.25)
    _format_date_axis(axes[0])

    axes[1].set_title(f'{title}: official rank by age group')
    axes[1].grid(alpha=0.25)
    _format_date_axis(axes[1])

    if not match_history.empty:
        match_history['match_date'] = pd.to_datetime(match_history['match_date'], errors='coerce')
        match_history = match_history.dropna(subset=['match_date']).sort_values('match_date')
        if pd.notna(prediction_date):
            match_history = match_history[match_history['match_date'] < prediction_date]
        if 'elo_pre' in match_history.columns:
            axes[2].plot(match_history['match_date'], match_history['elo_pre'], marker='o', linewidth=1.5, color='#3a9d62')
        if 'win' in match_history.columns:
            form = pd.to_numeric(match_history['win'], errors='coerce').rolling(10, min_periods=1).mean()
            axes[3].plot(match_history['match_date'], form, marker='o', linewidth=1.5, color='#b456c4')
            axes[3].set_ylim(-0.05, 1.05)

    axes[2].set_title(f'{title}: ELO before match')
    axes[2].grid(alpha=0.25)
    _format_date_axis(axes[2])

    axes[3].set_title(f'{title}: recent form, rolling win rate over last 10 matches')
    axes[3].grid(alpha=0.25)
    _format_date_axis(axes[3])

    plt.tight_layout()
    display(fig)
    plt.close(fig)



def _display_recent_matches(payload):
    specs = [
        ('Last 10 matches: ' + str(payload.get('player1_name')), payload.get('player1_last10')),
        ('Last 10 matches: ' + str(payload.get('player2_name')), payload.get('player2_last10')),
    ]
    rename = {
        'match_date': 'Date',
        'opponent_name': 'Opponent',
        'win': 'Result',
        'player_rank_pre': 'Player rank',
        'opponent_rank_pre': 'Opponent rank',
        'player_points_pre': 'Player points',
        'opponent_points_pre': 'Opponent points',
        'elo_pre': 'Player ELO',
        'elo_opp_pre': 'Opponent ELO',
        'tournament_name': 'Tournament',
        'tournament_city': 'City',
    }
    for title, records in specs:
        df = _to_frame(records)
        display(HTML(f'<h4>{html.escape(title)}</h4>'))
        if df.empty:
            display(HTML('<p>No matches found before the selected date.</p>'))
            continue
        if 'match_date' in df.columns:
            df['match_date'] = df['match_date'].map(_fmt_date)
        if 'win' in df.columns:
            df['win'] = df['win'].map(lambda x: 'Win' if x == 1 or x == 1.0 else 'Loss')
        keep = [col for col in rename if col in df.columns]
        display(df[keep].rename(columns=rename))



def _display_factor_contributions(payload):
    contrib = _to_frame(payload.get('factor_contributions'))
    if contrib.empty:
        display(HTML('<p>Forecast factor details are not available for this model.</p>'))
        return
    display(HTML('<h4>Main factors shifting player 1 probability</h4>'))
    show_cols = ['feature', 'group', 'probability_effect', 'value_player1_row', 'value_player2_row', 'meaning']
    show_cols = [col for col in show_cols if col in contrib.columns]
    table = contrib[show_cols].copy()
    if 'probability_effect' in table.columns:
        table['probability_effect'] = table['probability_effect'].map(lambda x: _fmt_pct(x, signed=True))
    display(table)

    plot_df = contrib.sort_values('probability_effect')
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
    colors = ['#ff8a4f' if v < 0 else '#4f7cff' for v in plot_df['probability_effect']]
    ax.barh(plot_df['feature'], plot_df['probability_effect'], color=colors)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Probability shift for player 1')
    ax.set_title('Main forecast drivers')
    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    display(fig)
    plt.close(fig)



def _display_common_opponents(payload):
    h2h = _to_frame(payload.get('h2h_detail'))
    common_summary = _to_frame(payload.get('common_summary'))
    common_detail = _to_frame(payload.get('common_detail'))
    signal = payload.get('common_signal') or {}
    edge = signal.get('common_opponents_edge')
    n_common = signal.get('n_common_opponents', 0)

    if edge is None or pd.isna(edge):
        signal_text = f'Common opponents before prediction date: {n_common}. Diagnostic edge is not calculated.'
    else:
        direction = payload.get('player1_name') if edge > 0 else payload.get('player2_name') if edge < 0 else 'neutral'
        signal_text = (
            f'Common opponents: {n_common}. Weighted edge: {_fmt_pct(edge, signed=True)}. '
            f'Signal direction: {html.escape(str(direction))}.'
        )
    display(HTML(f'<p>{signal_text}</p>'))

    display(HTML('<h4>Direct head-to-head matches</h4>'))
    if h2h.empty:
        display(HTML('<p>No direct head-to-head matches before prediction date.</p>'))
    else:
        if 'match_date' in h2h.columns:
            h2h['match_date'] = h2h['match_date'].map(_fmt_date)
        display(h2h)

    display(HTML('<h4>Common opponents summary</h4>'))
    if common_summary.empty:
        display(HTML('<p>No common opponents before prediction date.</p>'))
    else:
        summary = common_summary.copy()
        for col in ['player_a_winrate_vs_common', 'player_b_winrate_vs_common', 'winrate_edge_a_minus_b', 'weighted_edge']:
            if col in summary.columns:
                summary[col] = summary[col].map(lambda x: _fmt_pct(x, signed=col.endswith('edge') or col == 'winrate_edge_a_minus_b'))
        display(summary)

    display(HTML('<h4>Matches against common opponents</h4>'))
    if common_detail.empty:
        display(HTML('<p>No detailed common-opponent matches found.</p>'))
    else:
        detail = common_detail.copy()
        if 'match_date' in detail.columns:
            detail['match_date'] = detail['match_date'].map(_fmt_date)
        if 'win' in detail.columns:
            detail['win'] = detail['win'].map(lambda x: 'Win' if x == 1 or x == 1.0 else 'Loss')
        display(detail.head(120))



def run_prediction(_=None):
    with prediction_output:
        clear_output(wait=True)
        if not player1_box.value or not player2_box.value:
            display(HTML('<b>Select two players.</b>'))
            return
        if player1_box.value == player2_box.value:
            display(HTML('<b>Players must be different.</b>'))
            return
        try:
            payload = _run_prediction_cli([
                'predict',
                '--player1', player1_box.value,
                '--player2', player2_box.value,
                '--date', str(prediction_date_picker.value),
                '--age', AGE_CONTEXT.get(age_box.value, '__UNKNOWN_AGE__'),
                '--draw-type', '__UNKNOWN_DRAW__',
            ])
        except Exception as exc:
            display(HTML(f'<b>Prediction failed:</b><pre>{html.escape(str(exc))}</pre>'))
            return
        if not payload.get('ok'):
            display(HTML(f"<b>{html.escape(str(payload.get('message')))}</b>"))
            display(_to_frame(payload.get('player1_candidates')))
            display(_to_frame(payload.get('player2_candidates')))
            return

        _section('1. Player cards and factor dynamics', 'Metrics are calculated as of the expected match date, with no future matches included.')
        _display_player_cards(payload)
        _plot_player_history(payload.get('player1_history'), payload.get('player1_rating_history'), payload['player1_name'], payload.get('prediction_date'))
        _plot_player_history(payload.get('player2_history'), payload.get('player2_rating_history'), payload['player2_name'], payload.get('prediction_date'))

        _section('2. Last 10 matches for each player')
        _display_recent_matches(payload)

        _section('3. Win forecast and main drivers')
        display(HTML(_probability_bar(payload['player1_name'], payload['player2_name'], payload['p_player1_win'])))
        display(HTML(f"<p><b>Model:</b> {html.escape(str(payload.get('model_name')))}</p>"))
        _display_factor_contributions(payload)
        _plot_probability_timeline(payload)

        _section('4. Common opponents and direct meetings')
        _display_common_opponents(payload)


def reload_prediction_bundle(_=None):
    global player_names
    with prediction_output:
        clear_output(wait=True)
        try:
            payload = _run_prediction_cli(['players'])
            player_names = payload.get('players', [])
            player1_box.options = player_names
            player2_box.options = player_names
            prediction_status.value = f"<b>Prediction:</b> Loaded via project Python: {Path(PYTHON).name}; model: {payload.get('model_name')}; max match date: {payload.get('max_match_date')}"
            display(HTML('<b>Prediction bundle reloaded.</b>'))
        except Exception as exc:
            display(HTML(f'<b>Could not reload prediction bundle:</b> {html.escape(str(exc))}'))


predict_button.on_click(run_prediction)
reload_bundle_button.on_click(reload_prediction_bundle)

prediction_panel = widgets.VBox([
    widgets.HTML('<h3>Match forecast and analytics</h3>'),
    prediction_status,
    widgets.HBox([player1_box, player2_box]),
    widgets.HBox([prediction_date_picker, age_box, predict_button, reload_bundle_button]),
    prediction_output,
])

display(prediction_panel)